# Higgs planted targets: does the best attention reach track the task range?

Semi-synthetic targets with a **known range k** on the real Higgs follow graph (S3, `gnn_mech/social/plant.py`).
The real graph and the real node features stay (as nuisance inputs); planted channels are appended and the
target is an exact function of them:

- **`plant:shell_mean:k`** (sanity): y_v = mean of a planted N(0, 1) channel over the nodes at *exactly* k hops.
- **`plant:retrieve:k`** (main test): every node has a key code and a value; v has a query that matches the key of
  one (rarely a few) node(s) at exactly k hops and of *no* node closer than k; y_v = the matched value. The number
  of key values is scaled to the k-ball size, so the expected number of matches stays about 1 at every k.

The analytic range of both targets (Bamberger, from the exact Jacobian w.r.t. the planted channels) is k; step 9b
checks it. Full-graph transductive training (`data.sampler: full`): METIS parts only choose the labelled nodes.
Sweep `configs/sweeps/higgs_full_plant.yaml`: k ∈ {1, 2, 3} × both families × {mlp, gcn, hybrid α=0,
α=1 / α=0.5 Performer, α=0.5 radius-1 / radius-2 attention} × 2 model seeds (84 runs). k = 4 is excluded (the
4-ball is ~97% of this graph; see the sanity table). The radius-2 runs come last and may fail on the pair budget
(`model.attn_max_pairs`; the 2-hop pair list is ~3e8 pairs here): the sweep skips past them.

- **Resume.** After a disconnect, rerun every cell from the top, then the sweep cell: finished runs are skipped.
- **Use a GPU runtime.** The sanity table (step 9b) runs on CPU in about 5 minutes and is cached on Drive.


## 1. Settings (edit)

`BRANCH` defaults to `main`; set it to another branch to run that code instead.

In [ ]:
# EDIT: settings for this notebook
BRANCH = 'social-pipeline'          # git branch to run; change it to test a feature branch
USE_TOKEN = False        # True if the repo is private (needs the GITHUB_TOKEN Colab secret, see step 3)
DRIVE = '/content/drive/MyDrive/Thesis/gnn_mech'
DATA_ROOT = f'{DRIVE}/data'    # downloads and processed caches (kept on Drive, reused by every session)
OUT_DIR = f'{DRIVE}/runs'      # run folders (kept on Drive so runs resume after a disconnect)
LOCAL_DATA = '/content/data'   # fast local copy of the dataset used during training
DATASET = 'higgs-follow'
EXTRA = ''               # extra key=value overrides for every run (full-graph mode: no halo)

## 2. Check the GPU (Runtime > Change runtime type > T4 GPU)

In [ ]:
!nvidia-smi

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get the code

Clones the repo at `BRANCH`, or pulls if it is already there. If the repo is private, create a GitHub
personal access token (read access to the repo is enough), store it as a Colab secret named
`GITHUB_TOKEN` (key icon in the left sidebar) and set `USE_TOKEN = True` in step 1. Never paste the
token into the notebook itself.

In [ ]:
import os, subprocess

REPO_DIR = '/content/message-passing-in-gnns'
url = 'https://github.com/djdhawal/message-passing-in-gnns.git'
if USE_TOKEN:
    from google.colab import userdata
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/djdhawal/message-passing-in-gnns.git"

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

## 5. Install

In [ ]:
!pip install -q torch_geometric
!pip install -q -e .

## 6. Folders on Drive

In [ ]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
print('data on Drive:', DATA_ROOT)
print('runs on Drive:', OUT_DIR)

### Sweep file

In [ ]:
SWEEP = 'configs/sweeps/higgs_full_plant.yaml'
print('sweep:', SWEEP, '' if os.path.exists(SWEEP) else '(MISSING: check BRANCH)')

## 7. Data helpers

`find_source_dir` finds the dataset's folder under `<root>/social/`. `sync` copies only files that are
missing or newer (rsync). Nothing to edit here.

In [ ]:
import shutil
from pathlib import Path


def _source_candidates(name):
    # Folder names under <root>/social/ that may hold `name`; gnn_mech.social.sources decides when it can.
    cands = []
    try:
        from gnn_mech.social import sources
        for attr in ('source_of', 'source_name', 'source_for'):
            fn = getattr(sources, attr, None)
            if callable(fn):
                try:
                    cands.append(str(fn(name)))
                except Exception:
                    pass
        for attr in ('SOURCES', 'SOURCE_OF', 'DATASET_SOURCE', 'DATASETS'):
            table = getattr(sources, attr, None)
            if isinstance(table, dict) and name in table:
                v = table[name]
                if not isinstance(v, str):
                    v = v.get('source') if isinstance(v, dict) else getattr(v, 'source', None)
                if v:
                    cands.append(str(v))
    except Exception:
        pass
    cands.append(name)
    if name.startswith('twitch-') and name != 'twitch-gamers':
        cands += ['musae', 'twitch', 'twitch-musae', 'musae-twitch']
    if name.startswith('higgs-'):
        cands += ['higgs', 'higgs-twitter']
    if name.startswith('reddit-'):
        cands += ['reddit', 'reddit-hyperlinks', 'soc-redditHyperlinks']
    return list(dict.fromkeys(cands))


def find_source_dir(name, root, fetched=None):
    # Name of the folder <root>/social/<source>/ that holds `name`, or None if it does not exist yet.
    social = Path(root) / 'social'
    if isinstance(fetched, (str, os.PathLike)):
        try:
            rel = Path(fetched).resolve().relative_to(social.resolve())
            if rel.parts:
                return rel.parts[0]
        except Exception:
            pass
    for c in _source_candidates(name):
        if (social / c).is_dir():
            return c
    return None


def sync(src, dst):
    # Copy files that are missing or newer in src into dst; never deletes anything.
    os.makedirs(dst, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run(['rsync', '-a', '--update', f'{src}/', f'{dst}/'], check=True)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)

## 8. Download the dataset into Drive (once)

`gnn_mech.social.sources.fetch` downloads and parses the raw files into `DATA_ROOT/social/<source>/` and
skips files that already exist, so later sessions reuse them. If it fails, the run downloads the data
itself on first use.

In [ ]:
fetched = None
try:
    from gnn_mech.social import sources
    fetched = sources.fetch(DATASET, root=DATA_ROOT)
except Exception as e:
    print(f'[warn] fetch({DATASET!r}) failed: {e!r}')
SOURCE = find_source_dir(DATASET, DATA_ROOT, fetched)
print('source folder on Drive:', SOURCE)

## 9. Copy the dataset to local disk

Training reads from `LOCAL_DATA` (much faster than Drive). Only missing or newer files are copied.
Run outputs still go to `OUT_DIR` on Drive.

In [ ]:
if SOURCE:
    sync(f'{DATA_ROOT}/social/{SOURCE}', f'{LOCAL_DATA}/social/{SOURCE}')
    RUN_DATA = LOCAL_DATA
else:
    print('[warn] no dataset folder on Drive yet: this session reads and writes data on Drive directly')
    RUN_DATA = DATA_ROOT
COMMON = f'out_dir={OUT_DIR} data.root={RUN_DATA}'
print(COMMON)

In [ ]:
!du -sh {RUN_DATA}/social/* 2>/dev/null || echo 'nothing local yet'

## 9b. Planted-target sanity table (shell sizes, eligible nodes, analytic range, oracles)

Per k: shell sizes |S_k|, eligible nodes, how much of the graph the k-ball covers, and `shell_of_rest` (near 1:
the shell is everything outside the (k-1)-ball, so a shell mean is a constant minus inner-ball sums). Per target:
the number of retrieve keys K, realised matches, the analytic range (must equal k) and oracle R2s
(`inner_leak_r2`: y from the inner ball only; `global_key_match_r2`: y from all nodes with the query key at any
distance, i.e. what hop-unaware global retrieval could reach). Saved to `{DRIVE}/results/higgs_plant/`.
Computed on CPU in blocks of BFS frontiers (no N x N matrix); about 5 minutes for k = 1..4.

In [ ]:
import json, time, os
import pandas as pd
from gnn_mech.social import plant, sources

PLANT_RESULTS = f'{DRIVE}/results/higgs_plant'
os.makedirs(PLANT_RESULTS, exist_ok=True)
SANITY_KS = (1, 2, 3, 4)
spec_path = f'{PLANT_RESULTS}/plant_spec.json'
if os.path.exists(spec_path):
    with open(spec_path) as f:
        SANITY = json.load(f)
    print('loaded cached sanity table from', spec_path)
else:
    g = sources.load_graph(DATASET, RUN_DATA)
    t0 = time.time()
    SANITY = {'shell_stats': plant.shell_stats(g, SANITY_KS), 'targets': []}
    print(f'shell stats: {time.time() - t0:.0f}s')
    for fam in plant.FAMILIES:
        for k in SANITY_KS:
            t0 = time.time()
            Z, y, meta = plant.make(g, fam, k, 0)
            ar = plant.analytic_range(g, fam, k, 0, n_sample=128)
            SANITY['targets'].append({**plant.spec(meta), 'target': plant.name(fam, k),
                                      'analytic_range_hops': ar['range_hops'], 'analytic_range_min': ar['range_hops_min'],
                                      'analytic_range_max': ar['range_hops_max'], 'analytic_range_res': ar['range_res'],
                                      'seconds_total': round(time.time() - t0, 1)})
            print(f"{plant.name(fam, k)}: {time.time() - t0:.0f}s, analytic range {ar['range_hops']:.3f}")
    plant.clear_cache()
    with open(spec_path, 'w') as f:
        json.dump(SANITY, f, indent=1, default=str)

shells = pd.DataFrame([{'k': r['k'], 'eligible_frac': r['eligible_frac'], 'shell_median': r['shell']['median'],
                        'shell_mean': r['shell']['mean'], 'shell_p90': r['shell']['p90'], 'shell_max': r['shell']['max'],
                        'ball_frac_median': r['ball_frac']['median'], 'shell_of_rest': r['shell_of_rest']}
                       for r in SANITY['shell_stats']])
display(shells.round(3))
rows = []
for t in SANITY['targets']:
    o = t.get('oracle', {})
    rows.append({'target': t['target'], 'eligible_frac': t['eligible_frac'], 'n_keys': t.get('n_keys'),
                 'matches_mean': (t.get('matches') or {}).get('mean'), 'frac_unique': (t.get('matches') or {}).get('frac_unique'),
                 'analytic_range': t['analytic_range_hops'], 'inner_leak_r2': o.get('inner_leak_r2'),
                 'shell_value_mean_r2': o.get('shell_value_mean_r2'), 'global_key_match_r2': o.get('global_key_match_r2'),
                 'seconds': t['seconds_total']})
sanity_df = pd.DataFrame(rows)
sanity_df.to_csv(f'{PLANT_RESULTS}/sanity.csv', index=False)
shells.to_csv(f'{PLANT_RESULTS}/shell_stats.csv', index=False)
display(sanity_df.round(3))
bad = sanity_df[(sanity_df['analytic_range'] - sanity_df['target'].str.split(':').str[2].astype(float)).abs() > 1e-6]
print('analytic range == k for every target' if bad.empty else f'[warn] analytic range != k:\n{bad}')

## 10. Build the data cache (planted targets + full graph) (first session only) and save it to Drive

Loads every data config used by the sweep once. The first time, this partitions the graph and writes
`social/<source>/processed/<hash>.pt`; later sessions find it and skip the work. New files are then copied
back to Drive so the next session reuses them.

In [ ]:
import gc, json
from gnn_mech.data import load_dataset

from gnn_mech.sweep import load_sweep
try:
    DATA_CFGS = [(r.name, r.cfg) for r in load_sweep(SWEEP, [f'out_dir={OUT_DIR}', f'data.root={RUN_DATA}'] + ([EXTRA] if EXTRA else []))[0]]
except Exception as e:
    DATA_CFGS = []
    print(f'[warn] could not read {SWEEP}: {e!r}')
seen = set()
for label, c in DATA_CFGS:
    d = c.to_dict()['data']
    d.pop('root', None)
    key = json.dumps(d, sort_keys=True, default=str)
    if key in seen:
        continue
    seen.add(key)
    try:
        tr, va, te, info = load_dataset(c.data)
        print(f'{label}: {len(tr)} / {len(va)} / {len(te)} subgraphs (train / val / test), '
              f'task {info.task}, feat_dim {getattr(info, "feat_dim", "?")}, '
              f'nodes per graph {tr[0].num_nodes}, labelled train nodes {int(tr[0].y_mask.sum())}')
        del tr, va, te
    except Exception as e:
        print(f'[warn] could not prepare data for {label}: {e!r} (the run will try again)')
gc.collect()

if SOURCE is None:
    SOURCE = find_source_dir(DATASET, RUN_DATA)
if RUN_DATA != DATA_ROOT and SOURCE:
    sync(f'{RUN_DATA}/social/{SOURCE}', f'{DATA_ROOT}/social/{SOURCE}')
    print('cache saved to Drive:', f'{DATA_ROOT}/social/{SOURCE}')

## 11. Dry run: the run list, each run's status and the estimated remaining hours

In [ ]:
!python -m gnn_mech.sweep {SWEEP} --dry-run {COMMON} {EXTRA}

## 12. Run the sweep (rerun this cell after a disconnect to resume)

In [ ]:
!python -m gnn_mech.sweep {SWEEP} {COMMON} {EXTRA}

### Save any new data files back to Drive

In [ ]:
if RUN_DATA != DATA_ROOT:
    SOURCE = SOURCE or find_source_dir(DATASET, RUN_DATA)
    if SOURCE:
        sync(f'{RUN_DATA}/social/{SOURCE}', f'{DATA_ROOT}/social/{SOURCE}')
        print('synced to Drive')

## 13. Results: test R2 and measured range vs planted k

Left: test R2 vs planted k, one line per variant (mean ± std over model seeds). Right: the measured node-level
range of each trained model (`measurements.json` → `summary.range_node.range_hops.rows.mean`, Bamberger, hops)
vs planted k, with the analytic range (= k) as the dashed diagonal. Saved under `{DRIVE}/results/higgs_plant/`.

In [ ]:
import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VARIANT_ORDER = ['mlp', 'gcn', 'alpha=0', 'alpha=1 performer', 'alpha=0.5 performer', 'radius 1', 'radius 2']


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def variant_of(model):
    arch = model.get('arch')
    if arch != 'hybrid':
        return arch
    if float(model.get('alpha', 0.5)) == 0.0:
        return 'alpha=0'
    if model.get('attn') == 'radius' and int(model.get('attn_radius', -1)) >= 0:
        return f"radius {int(model['attn_radius'])}"
    return f"alpha={float(model['alpha']):g} {model.get('attn', 'softmax')}"


def measured_range(meas, key='range_hops'):
    try:
        return meas['summary']['range_node'][key]['rows']['mean']
    except (KeyError, TypeError):
        return None


def collect_plant(out_dir, exp='higgs_full_plant'):
    # One row per finished run of `exp` with a planted target (config.json + metrics.json; measurements optional).
    rows = []
    for cfg_path in Path(out_dir).glob(f'{exp}/*/seed*/config.json'):
        run = cfg_path.parent
        cfg, m = read_json(cfg_path), read_json(run / 'metrics.json')
        target = ((cfg or {}).get('data') or {}).get('target', '')
        if not cfg or not m or not isinstance(m.get('test'), dict) or not target.startswith('plant:'):
            continue
        _, family, k = target.split(':')[:3]
        meas = read_json(run / 'measurements.json') or {}
        rows.append({'target': target, 'family': family, 'k': int(k), 'variant': variant_of(cfg.get('model', {})),
                     'seed': cfg.get('seed'), 'test_r2': m['test'].get('r2'), 'val_r2': (m.get('val') or {}).get('r2'),
                     'range_hops': measured_range(meas, 'range_hops'), 'range_res': measured_range(meas, 'range_res'),
                     'best_epoch': m.get('best_epoch'), 'minutes': (m.get('seconds') or 0) / 60, 'path': str(run)})
    return pd.DataFrame(rows)


def plot_plant(df, out):
    os.makedirs(out, exist_ok=True)
    df.to_csv(f'{out}/runs.csv', index=False)
    agg = (df.groupby(['family', 'k', 'variant'])
             .agg(n=('test_r2', 'size'), r2=('test_r2', 'mean'), r2_std=('test_r2', 'std'),
                  range_hops=('range_hops', 'mean'), range_std=('range_hops', 'std'), minutes=('minutes', 'sum'))
             .reset_index().fillna({'r2_std': 0.0, 'range_std': 0.0}))
    agg.to_csv(f'{out}/summary.csv', index=False)
    fams = sorted(agg['family'].unique())
    variants = [v for v in VARIANT_ORDER if v in set(agg['variant'])] + sorted(set(agg['variant']) - set(VARIANT_ORDER))
    fig, axes = plt.subplots(len(fams), 2, figsize=(11, 3.8 * len(fams)), squeeze=False)
    for row, fam in zip(axes, fams):
        sub = agg[agg['family'] == fam]
        for i, v in enumerate(variants):
            s = sub[sub['variant'] == v].sort_values('k')
            if s.empty:
                continue
            row[0].errorbar(s['k'], s['r2'], yerr=s['r2_std'], marker='o', capsize=3, color=f'C{i}', label=v)
            r = s.dropna(subset=['range_hops'])
            if len(r):
                row[1].errorbar(r['k'], r['range_hops'], yerr=r['range_std'], marker='o', capsize=3, color=f'C{i}', label=v)
        ks = sorted(sub['k'].unique())
        row[1].plot(ks, ks, ls='--', color='grey', label='analytic (= k)')
        row[0].set(title=f'{fam}: test R2', xlabel='planted k (hops)', ylabel='test R2', xticks=ks)
        row[1].set(title=f'{fam}: measured node-level range', xlabel='planted k (hops)', ylabel='range (hops)', xticks=ks)
        row[0].legend(fontsize=7)
        row[1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(f'{out}/r2_and_range_vs_k.png', dpi=150)
    return agg, fig


RESULTS = f'{DRIVE}/results/higgs_plant'
plant_df = collect_plant(OUT_DIR)
if plant_df.empty:
    print('no finished planted runs yet')
else:
    agg, fig = plot_plant(plant_df, RESULTS)
    display(agg.round(4))
    plt.show()
    # Gate S3: for the best model per (family, k), does the measured range rise with k?
    from scipy.stats import spearmanr
    for fam, sub in agg.dropna(subset=['range_hops']).groupby('family'):
        best = sub.loc[sub.groupby('k')['r2'].idxmax()]
        if len(best) >= 3:
            rho = spearmanr(best['k'], best['range_hops']).correlation
            print(f'{fam}: best variant per k {list(best["variant"])}, Spearman(k, measured range) = {rho:.2f}')